In [8]:
!pip install pinecone pinecone-text pinecone-notebooks

   ---------------------------------------- 0.0/587.6 kB ? eta -:--:--
   ---------------------------------------- 587.6/587.6 kB 10.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   -------------------- ------------------- 1/2 [pinecone]
   -------------------- ------------------- 1/2 [pinecone]
   -------------------- ------------------- 1/2 [pinecone]
   -------------------- ------------------- 1/2 [pinecone]
   -------------------- ------------------- 1/2 [pinecone]
   -------------------- ------------------- 1/2 [pinecone]
   -------------------- ----------------

In [1]:
import os

from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv('PINECONE_API_KEY')

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [3]:
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"

## intialize the Pinecone client
pc = Pinecone(api_key= api_key)

if index_name not in pc.list_indexes().names():
  pc.create_index(
    name= index_name,
    dimension=  384, ## dimension of dense vector: BECUASE hugging face embedding technique uses sentence-transformer which has default dim of 384
    metric= 'dotproduct', ## used for sparse matrix values
    spec= ServerlessSpec(cloud= 'aws', region= "us-east-1")
  )

In [4]:
index= pc.Index(index_name)
index

d:\DATA SCIENCE ML AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Semantic embeddings

In [5]:
## vector embeddings and sparse matrix
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model= "all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Sparse Matrix Conversion

In [6]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

In [7]:
sentences= [
  "In 2003, I visited Paris",
  "In 2022, I visited New York",
  "In 2021, I visited New Orleans",
]

### Applying TF-IDF to these sentences:

In [ ]:
bm25_encoder.fit(sentences)
bm25_encoder.dump("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 103.43it/s]


In [10]:
bm25_encoder = BM25Encoder().load("bm25_values.json")

In [11]:
retriever = PineconeHybridSearchRetriever(
  embeddings= embeddings, ## Semantic Search
  sparse_encoder= bm25_encoder, ## Syntactic Search
  index= index
)

In [12]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000023276845C10>, index=<pinecone.db_data.index.Index object at 0x0000023224BE4E10>)

In [14]:
retriever.add_texts(sentences)

100%|██████████| 1/1 [00:02<00:00,  2.98s/it]


In [19]:
retriever.invoke("What city did I visit first?")

[Document(metadata={'score': 0.230018824}, page_content='In 2003, I visited Paris'),
 Document(metadata={'score': 0.244533196}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.257887483}, page_content='In 2021, I visited New Orleans')]